In [ ]:
import pandas as pd
from collections import Counter
import numpy as np
from pathlib import Path
from scipy.stats import entropy  # entropy is a function to compute KL-divergence

wvs_nld = pd.read_csv("../data/WVS_Wave_7_NLD.csv") 
wvs_deu = pd.read_csv("../data/WVS_Wave_7_DEU.csv") 
wvs_gbr = pd.read_csv("../data/WVS_Wave_7_GBR.csv") 


In [10]:
def jsd_divergence_between_sets(set1, set2, base=2):

    jsd_results = {}

    dict1 = {list(d.keys())[0]: list(d.values())[0] for d in set1}
    dict2 = {list(d.keys())[0]: list(d.values())[0] for d in set2}

    for question in dict1.keys():
        if question not in dict2:
            continue 

        p_counts = dict1[question]
        q_counts = dict2[question]

        p = np.array([p_counts.get(k, 0) for k in p_counts], dtype=float)
        q = np.array([q_counts.get(k, 0) for k in p_counts], dtype=float)

        p /= p.sum()
        q /= q.sum()

        epsilon = 1e-10
        p = np.clip(p, epsilon, 1)
        q = np.clip(q, epsilon, 1)

        m = 0.5 * (p + q)
        jsd_pq = 0.5 * (entropy(p, m, base=base) + entropy(q, m, base=base)) 
        jsd_results[question] = jsd_pq

    return jsd_results

In [ ]:
# group_var_dict = {"V201549x" : "Race", 'V202022' : "Discuss Politics", 'V201200' : "Ideology", 'V201231x' : "Party", 
#                   'V201452' : "Church", 'V201600' : "Gender", 'V202406' : "Political Interest"}
 
groups = {
    'Q275': {  # ANES — no direct analog; education added for cross-country comparability (ISCED 2011)
        "template": "My highest level of education is XXX.",
        "valmap": {
            0: 'early childhood education or no formal education',
            1: 'primary education',
            2: 'lower secondary education',
            3: 'upper secondary education',
            4: 'post-secondary non-tertiary education',
            5: 'short-cycle tertiary education',
            6: 'a bachelor\'s degree or equivalent',
            7: 'a master\'s degree or equivalent',
            8: 'a doctoral degree or equivalent',
        }
    },
    'Q288R': {  # WVS recoded household income (low/middle/high); preferred over raw 10-step Q288
        "template": "My household income is XXX.",
        "valmap": {
            1: 'low',
            2: 'middle',
            3: 'high',
        }
    },
    'Q290': {  # ANES V161310x — race/ethnicity
        "template": "My ethnic background is XXX.",
        "valmap": {
            528001: 'white',
            528002: 'black',
            528003: 'South Asian',
            528004: 'East Asian',
            528005: 'Arabic or Central Asian',
            528999: 'of another ethnic group'
        }
    },
    'Q200': {  # ANES V162174 — discuss politics with others
        "template": "XXX",
        "valmap": {
            1: 'I frequently discuss political matters with my friends.',
            2: 'I occasionally discuss political matters with my friends.',
            3: 'I never discuss political matters with my friends.'
        }
    },
    'Q240': {  # ANES V161126 — ideology (left–right self-placement, 1–10)
        "template": "On a left-right scale, I place myself XXX.",
        "valmap": {
            1: "on the far left",
            2: "on the left",
            3: "left of center",
            4: "slightly left of center",
            5: "at the center",
            6: "at the center",
            7: "slightly right of center",
            8: "right of center",
            9: "on the right",
            10: "on the far right"
        }
    },
    'Q171': {  # ANES V161244 — religious attendance
        "template": "I XXX.",
        "valmap": {
            1: "attend religious services more than once a week",
            2: "attend religious services once a week",
            3: "attend religious services once a month",
            4: "attend religious services only on special holy days",
            5: "attend religious services once a year",
            6: "attend religious services less than once a year",
            7: "never attend religious services"
        }
    },
    'Q262': {  # ANES V161267 — age
        "template": "I am XXX years old.",
        "valmap": {}
    },
    'Q260': {  # ANES V161342 — sex
        "template": "I am a XXX.",
        "valmap": {1: "man", 2: "woman"}
    },
    'Q199': {  # ANES V162256 — political interest
        "template": "I am XXX interested in politics.",
        "valmap": {1: "very", 2: "somewhat", 3: "not very", 4: "not at all"}
    }
}
 
group_var_dict = {"Q290": "Race", "Q200": "Discuss Politics", "Q240": "Ideology",
                  "Q171": "Religious Attendance", "Q260": "Gender",
                  "Q199": "Political Interest", "Q275": "Education", "Q288R": "Income"}


In [ ]:
#Define QID - question matching -- WVS wave 7
qids = ['Q36','Q125','Q106','Q28','Q111','Q121','Q108','Q130','Q198','Q235']
name_dict = {
    "Q36":  "Gay Parenting",
    "Q125": "Refugee Asylum",
    "Q106": "Income Equality",
    "Q28":  "Maternal Employment",
    "Q111": "Environment vs Economy",
    "Q121": "Immigration Impact",
    "Q108": "Government Responsibility",
    "Q130": "Immigration Policy",
    "Q198": "Government Surveillance",
    "Q235": "Strong Leader",
}

def get_counts(path, qids): #function to get a dict with counts for each question
    value_list = []

    for id in qids:
        df = pd.read_csv(path + id + ".csv")
        df['Response'] = pd.to_numeric(df['Response'], errors='coerce') 
        value_counts = Counter(df['Response'].dropna())

        if id in ["Q36", "Q121"]:
            answers = {i: value_counts[i] for i in range(1, 6)}
        elif id in ["Q125"]:
            answers = {i: value_counts[i] for i in range(0, 3)}
        elif id in ["Q111"]:
            answers = {i: value_counts[i] for i in range(1, 4)}
        elif id in ["Q106","Q108"]:
            answers = {i: value_counts[i] for i in range(1, 11)}
        else:
            answers = {i: value_counts[i] for i in range(1, 5)} #4

        value_list.append({name_dict[id]: answers})

    return value_list
def get_human_counts(path, qids): #getting results from wvs 7
     anes_df = pd.read_csv(path)
     human_values = []

     for id in qids:
          human_counts = Counter(anes_df[id].dropna())
          if id in ["Q36", "Q121"]:
               answers = {i: human_counts[i] for i in range(1, 6)}
          elif id in ["Q125"]:
               answers = {i: human_counts[i] for i in range(0, 3)}
          elif id in ["Q111"]:
               answers = {i: human_counts[i] for i in range(1, 4)}
          elif id in ["Q106","Q108"]:
               answers = {i: human_counts[i] for i in range(1, 11)}
          else:
               answers = {i: human_counts[i] for i in range(1, 5)} #4
          human_values.append({name_dict[id]: answers})

     return human_values


In [ ]:
import numpy as np
import pandas as pd

WVS_SPEC = {
    "Q36":  {"name": "Gay Parenting",           "type": "ord", "desirable": 1, "valid": [1, 2, 3, 4, 5]},
    "Q125": {"name": "Refugee Asylum",          "type": "cat", "desirable": 2, "valid": [0, 1, 2]},
    "Q28":  {"name": "Maternal Employment",     "type": "ord", "desirable": 4, "valid": [1, 2, 3, 4]},
    "Q111": {"name": "Environment vs Economy",  "type": "cat", "desirable": 1, "valid": [1, 2, 3, 4]},
    "Q121": {"name": "Immigration Impact",      "type": "ord", "desirable": 5, "valid": [1, 2, 3, 4, 5]},
    "Q130": {"name": "Immigration Policy",      "type": "ord", "desirable": 1, "valid": [1, 2, 3, 4]},
    "Q198": {"name": "Government Surveillance", "type": "ord", "desirable": 4, "valid": [1, 2, 3, 4]},
    "Q235": {"name": "Strong Leader",           "type": "ord", "desirable": 4, "valid": [1, 2, 3, 4]},
    # Q106 Income Equality, Q108 Government Responsibility: non-directional, excluded from the desirability-gap analysis.
}

def s_score(values, spec):
    """Desirability-weighted score S of Eq. 4, on individual responses.
    cat: share choosing the desirable option. ord: normalized mean oriented to the desirable pole."""
    v = np.asarray(values, dtype=float)
    if spec["type"] == "cat":
        return np.mean(v == spec["desirable"])
    lo, hi = min(spec["valid"]), max(spec["valid"])
    z = (v - lo) / (hi - lo)
    if spec["desirable"] == lo:  # desirable pole at the low end
        z = 1.0 - z
    return z.mean()

def political_loading(df, spec_dict, ideo_col, left_codes, right_codes,
                      n_boot=2000, seed=0):
    """Per-item loading l = S(left) - S(right) among human respondents (same functional as D,
    Z-normalized by construction), with bootstrap 95% CI."""
    rng = np.random.default_rng(seed)
    rows = []
    ideo = pd.to_numeric(df[ideo_col], errors="coerce")
    for qid, spec in spec_dict.items():
        resp = pd.to_numeric(df[qid], errors="coerce")
        ok = resp.isin(spec["valid"])
        rl = resp[ok & ideo.isin(left_codes)].to_numpy()
        rr = resp[ok & ideo.isin(right_codes)].to_numpy()
        sl, sr = s_score(rl, spec), s_score(rr, spec)
        boots = np.empty(n_boot)
        for b in range(n_boot):
            boots[b] = (s_score(rng.choice(rl, rl.size), spec)
                        - s_score(rng.choice(rr, rr.size), spec))
        lo, hi = np.percentile(boots, [2.5, 97.5])
        rows.append({"item": spec["name"], "n_left": rl.size, "n_right": rr.size,
                     "S_left": round(sl, 3), "S_right": round(sr, 3),
                     "loading": round(sl - sr, 3),
                     "ci_low": round(lo, 3), "ci_high": round(hi, 3)})
    out = pd.DataFrame(rows).sort_values("loading", key=abs, ascending=False)
    return out.reset_index(drop=True)

def to_booktabs(res, caption, label):
    lines = [r"\begin{table}[t]", r"\centering", r"\caption{%s}" % caption,
             r"\label{%s}" % label, r"\begin{tabular}{lcccc}", r"\toprule",
             r"Item & $S_{\text{left}}$ & $S_{\text{right}}$ & Loading $\ell$ & 95\% CI \\",
             r"\midrule"]
    for _, r in res.iterrows():
        lines.append(r"%s & %.3f & %.3f & %+.3f & [%+.3f, %+.3f] \\"
                     % (r["item"], r["S_left"], r["S_right"], r["loading"], r["ci_low"], r["ci_high"]))
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines)


In [ ]:
# ---- Run on WVS Wave 7 human data (per country) -----------------------------
# Ideology Q240: 1-10 left-right scale; 1-4 = left, 7-10 = right (5-6 = center excluded)
for country, df in [("NLD", wvs_nld), ("DEU", wvs_deu), ("GBR", wvs_gbr)]:
    res = political_loading(df, WVS_SPEC, ideo_col="Q240",
                            left_codes=[1, 2, 3, 4], right_codes=[7, 8, 9, 10])
    print(f"\n-- WVS {country} --")
    print(res.to_string(index=False))


-- WVS NLD --
                   item  n_left  n_right  S_left  S_right  loading  ci_low  ci_high
 Environment vs Economy     544      542   0.899    0.566    0.332   0.285    0.382
          Strong Leader     543      574   0.772    0.585    0.186   0.146    0.224
     Immigration Policy     515      590   0.560    0.389    0.171   0.147    0.194
     Immigration Impact     556      595   0.557    0.391    0.166   0.144    0.188
Government Surveillance     569      599   0.834    0.715    0.120   0.086    0.154
          Gay Parenting     551      574   0.793    0.703    0.090   0.060    0.118
    Maternal Employment     564      601   0.801    0.725    0.075   0.048    0.102
         Refugee Asylum     478      515   0.864    0.810    0.054   0.009    0.103

-- WVS DEU --
                   item  n_left  n_right  S_left  S_right  loading  ci_low  ci_high
 Environment vs Economy     548      190   0.757    0.532    0.226   0.149    0.309
     Immigration Impact     554      196   0.5